# 2D elastic FWI with parallel sources, `AutomatedAdjoint`, and ROL

This notebook builds a **2D isotropic elastic** FWI example without using `spyro.FullWaveformInversion`. The workflow follows the idea of the `fwi_elastic.py` script: each MPI ensemble member solves one source, the local functional is recorded by automatic adjoint, the source contributions are summed with `EnsembleReducedFunctional`, and the optimization is solved with ROL.

The current geometry uses 4 sources and 50 receivers at the top of the domain. Both the true model and the initial model are horizontal layered models. The control parameters are:

$$m = (v_p, v_s),$$

while the density `rho` is kept fixed.

Main objects used here:

- `spyro.IsotropicWave`, for the isotropic elastic wave solve;
- `wave.enable_automated_adjoint()`, which creates and configures Spyro's `AutomatedAdjoint` API;
- `firedrake.adjoint.EnsembleReducedFunctional`, to sum the local source functionals;
- `pyadjoint.MinimizationProblem` and `firedrake.adjoint.ROLSolver`, to solve the optimization problem with ROL.

## How to run

This notebook should be converted to a Python script and run with MPI. A regular Jupyter session usually uses only one rank. The number of ranks must be divisible by `NUMBER_OF_SOURCES`.

From the repository root, you can run with the firedrake venv activated the following script:

```bash
pip install pyroltrilinos
python -m jupyter nbconvert --to python notebook_tutorials/parallel_sources_layered_fwi.ipynb
mpiexec -n 4 python notebook_tutorials/parallel_sources_layered_fwi.py
```

Use `-n 4` for one rank per source. Use `-n 8`, `-n 12`, etc. to combine source parallelism with internal spatial parallelism.

In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt
import firedrake as fire
import spyro

from firedrake.adjoint import EnsembleReducedFunctional, ROLSolver
from mpi4py import MPI
from pyadjoint import Control, MinimizationProblem, taylor_test
from spyro.domains.space import create_function_space
from spyro.solvers import AutomatedAdjoint


REQUIRED_NUMBER_OF_SOURCES = 4
NUMBER_OF_SOURCES = 4
NUMBER_OF_RECEIVERS = 50

world = MPI.COMM_WORLD
WRITE_VTK_OUTPUT = True
VTK_OUTPUT_ENSEMBLE_RANK = 0


def write_vtk_on_ensemble_root(wave, filename, *functions):
    if WRITE_VTK_OUTPUT and wave.comm.ensemble_comm.rank == VTK_OUTPUT_ENSEMBLE_RANK:
        fire.VTKFile(str(filename), comm=wave.comm.comm).write(*functions)


if NUMBER_OF_SOURCES != REQUIRED_NUMBER_OF_SOURCES:
    raise RuntimeError(
        "This tutorial was built for exactly "
        f"{REQUIRED_NUMBER_OF_SOURCES} parallel sources. "
        f"Received value: {NUMBER_OF_SOURCES}."
    )

if world.size % NUMBER_OF_SOURCES != 0:
    raise RuntimeError(
        f"Run with a number of ranks divisible by {NUMBER_OF_SOURCES}. "
        f"Received {world.size} ranks for {NUMBER_OF_SOURCES} sources."
    )

if world.rank == 0:
    print(f"MPI ranks: {world.size}")
    print(f"Sources: {NUMBER_OF_SOURCES}")
    print(f"Ranks per source: {world.size // NUMBER_OF_SOURCES}")


## 2. Problem setup

The domain is `1 km x 1 km`. In 2D, Spyro uses coordinates `(z, x)`, with `z = 0` at the top and negative values at depth. The 4 sources and 50 receivers are placed near the top. The elastic source below injects force in the horizontal `x` direction.

VTK output is restricted to the root ensemble member (`VTK_OUTPUT_ENSEMBLE_RANK = 0`) and uses `comm=wave.comm.comm`, so all spatial ranks in that ensemble member participate in the parallel write.

In [ ]:
output_dir = Path("results/parallel_layered_elastic_fwi_auto_adjoint")
output_dir.mkdir(parents=True, exist_ok=True)

length_z = 1.0
length_x = 1.0
degree = 4
frequency = 7.0
final_time = 1.0
dt = 0.001
edge_length = 0.05
rho_value = 1.0

source_locations = spyro.create_transect(
    (-0.10, 0.1),
    (-0.10, 0.9),
    NUMBER_OF_SOURCES,
)
receiver_locations = spyro.create_transect(
    (-0.12, 0.15),
    (-0.12, 0.85),
    NUMBER_OF_RECEIVERS,
)

if len(source_locations) != REQUIRED_NUMBER_OF_SOURCES:
    raise RuntimeError(
        "This tutorial geometry must contain exactly "
        f"{REQUIRED_NUMBER_OF_SOURCES} sources. "
        f"Created {len(source_locations)}."
    )

if len(receiver_locations) != NUMBER_OF_RECEIVERS:
    raise RuntimeError(
        "This tutorial geometry must contain exactly "
        f"{NUMBER_OF_RECEIVERS} receivers. "
        f"Created {len(receiver_locations)}."
    )

if world.rank == 0:
    print("Sources:")
    for source in source_locations:
        print(f"  {source}")
    print(f"Receivers: {len(receiver_locations)}")


## 3. `IsotropicWave` dictionary

Source parallelism is configured with `parallelism["type"] = "automatic"`. Internally, Spyro creates a `firedrake.Ensemble` and distributes source groups through `wave.comm`. This same `wave.comm` is passed later to `EnsembleReducedFunctional`.

The `density`, `p_wave_velocity`, and `s_wave_velocity` fields are filled after mesh creation because we want to pass `firedrake.Function` objects for `v_p` and `v_s`.

In [ ]:
def make_dictionary():
    return {
        "options": {
            "cell_type": "Q",
            "variant": "lumped",
            "degree": degree,
            "dimension": 2,
        },
        "parallelism": {
            "type": "automatic",
        },
        "mesh": {
            "length_z": length_z,
            "length_x": length_x,
            "length_y": 0.0,
            "mesh_file": None,
            "mesh_type": "firedrake_mesh",
        },
        "acquisition": {
            "source_type": "ricker",
            "source_locations": source_locations,
            "frequency": frequency,
            "delay": 0.2,
            "delay_type": "time",
            "amplitude": np.array([0.0, 1.0]),
            "receiver_locations": receiver_locations,
            "use_vertex_only_mesh": False,
        },
        "time_axis": {
            "initial_time": 0.0,
            "final_time": final_time,
            "dt": dt,
            "output_frequency": 100,
            "gradient_sampling_frequency": 1,
        },
        "visualization": {
            "forward_output": False,
            "forward_output_filename": str(output_dir / "forward_output.pvd"),
            "fwi_velocity_model_output": False,
            "velocity_model_filename": None,
            "gradient_output": False,
            "gradient_filename": str(output_dir / "gradient.pvd"),
            "adjoint_output": False,
            "adjoint_filename": None,
            "debug_output": False,
        },
        "synthetic_data": {
            "type": "object",
            "density": rho_value,
            "p_wave_velocity": None,
            "s_wave_velocity": None,
            "real_velocity_file": None,
        },
    }


## 4. Horizontal layered elastic models

We define horizontal layers for `v_p` and `v_s`. The density is kept fixed. To keep the material physical, we require `v_p > sqrt(2) v_s`, which is equivalent to `lambda > 0` when `rho` is constant.

In [ ]:
def horizontal_layers(mesh_z, interface_1, interface_2, v_top, v_mid, v_bottom):
    return fire.conditional(
        mesh_z > interface_1,
        v_top,
        fire.conditional(mesh_z > interface_2, v_mid, v_bottom),
    )


def true_vp(mesh_z):
    return horizontal_layers(mesh_z, -0.30, -0.6, 2.20, 2.80, 3.40)


def true_vs(mesh_z):
    return horizontal_layers(mesh_z, -0.3, -0.6, 1.10, 1.40, 1.70)


def initial_vp(mesh_z):
    return horizontal_layers(mesh_z, -0.3, -0.6, 2., 2., 2.)


def initial_vs(mesh_z):
    return horizontal_layers(mesh_z, -0.3, -0.6, 1.10, 1.10, 1.10)


def make_scalar_space(wave):
    return create_function_space(wave.mesh, wave.method, wave.degree, dim=1)


def make_parameter_function(wave, expression, name):
    V0 = make_scalar_space(wave)
    return fire.Function(V0, name=name).interpolate(expression)


## 5. Synthetic observed data

We create observed data by solving the elastic forward problem with the true model. The resulting shot record is vector-valued, with one component per spatial dimension.

In [ ]:
wave_true = spyro.IsotropicWave(dictionary=make_dictionary())
wave_true.set_mesh(input_mesh_parameters={"edge_length": edge_length})


def _print_mpi_topology(wave, tag):
    """Print the wave object ensemble topology in world-rank order.

    This helps verify that 'parallelism = automatic' is actually splitting
    sources across ranks. For N_sources sources and N_world ranks, the expected
    topology is ensemble_comm.size == N_sources and comm.size == N_world/N_sources.
    """
    # Gather topology information on world rank 0.
    info = (
        world.rank,
        wave.comm.ensemble_comm.rank,
        wave.comm.ensemble_comm.size,
        wave.comm.comm.rank,
        wave.comm.comm.size,
        list(getattr(wave, "shot_ids_per_propagation", [])),
    )
    gathered = world.gather(info, root=0)
    if world.rank == 0:
        print(f"\n[MPI-topology {tag}]")
        print(
            f"  world.size = {world.size}, expected per source = "
            f"{world.size // NUMBER_OF_SOURCES}"
        )
        for w, er, es, cr, cs, shots in gathered:
            print(
                f"  rank w={w}: ensemble({er}/{es}), spatial({cr}/{cs}), "
                f"shots={shots}"
            )
        print()


_print_mpi_topology(wave_true, "wave_true")

vp_true = make_parameter_function(wave_true, true_vp(wave_true.mesh_z), "vp_true")
vs_true = make_parameter_function(wave_true, true_vs(wave_true.mesh_z), "vs_true")

wave_true.input_dictionary["synthetic_data"]["p_wave_velocity"] = vp_true
wave_true.input_dictionary["synthetic_data"]["s_wave_velocity"] = vs_true
_t_fwd_true = time.perf_counter()
wave_true.forward_solve()
_t_fwd_true = time.perf_counter() - _t_fwd_true

observed_data = wave_true.forward_solution_receivers

# Report the forward time on each rank to detect imbalance.
_fwd_times = world.gather(_t_fwd_true, root=0)
if world.rank == 0:
    print(f"\n[timing] true forward per rank (s): {[f'{t:.2f}' for t in _fwd_times]}")

if wave_true.comm.comm.rank == 0 and wave_true.comm.ensemble_comm.rank == 0:
    print("Observed data shape:", np.asarray(observed_data).shape)
write_vtk_on_ensemble_root(wave_true, output_dir / "vp_true.pvd", vp_true)
write_vtk_on_ensemble_root(wave_true, output_dir / "vs_true.pvd", vs_true)


## 6. Annotated forward solve with `v_p` and `v_s` controls

We now build the initial model, register the observed shot record in `real_shot_record`, and activate `AutomatedAdjoint` through `enable_automated_adjoint()`.

By default, `IsotropicWave` may choose derived elastic controls such as `rho`, `lambda`, and `mu`. Here we explicitly overwrite the controls with `v_p` and `v_s`, while keeping `rho` fixed:

```python
wave_guess.automated_adjoint.controls = [vp_guess, vs_guess]
```

In [ ]:
wave_guess = spyro.IsotropicWave(dictionary=make_dictionary())
wave_guess.set_mesh(input_mesh_parameters={"edge_length": edge_length})

vp_guess = make_parameter_function(wave_guess, initial_vp(wave_guess.mesh_z), "vp_guess")
vs_guess = make_parameter_function(wave_guess, initial_vs(wave_guess.mesh_z), "vs_guess")

wave_guess.input_dictionary["synthetic_data"]["p_wave_velocity"] = vp_guess
wave_guess.input_dictionary["synthetic_data"]["s_wave_velocity"] = vs_guess
wave_guess.real_shot_record = observed_data

wave_guess.enable_automated_adjoint()
wave_guess.automated_adjoint.controls = [vp_guess, vs_guess]
_print_mpi_topology(wave_guess, "wave_guess")
_t_fwd_guess = time.perf_counter()
wave_guess.forward_solve()
_t_fwd_guess = time.perf_counter() - _t_fwd_guess

_fwd_times = world.gather(_t_fwd_guess, root=0)
if world.rank == 0:
    print(f"\n[timing] initial annotated forward per rank (s): {[f'{t:.2f}' for t in _fwd_times]}")
    print(f"[tape  ] number of blocks = {len(wave_guess.automated_adjoint._tape.get_blocks())}")

if wave_guess.comm.comm.rank == 0 and wave_guess.comm.ensemble_comm.rank == 0:
    print("Annotated local functional:", wave_guess.functional_value)
write_vtk_on_ensemble_root(wave_guess, output_dir / "vp_initial.pvd", vp_guess)
write_vtk_on_ensemble_root(wave_guess, output_dir / "vs_initial.pvd", vs_guess)


## 7. `EnsembleReducedFunctional`

First we use Spyro's `AutomatedAdjoint` API to create the local reduced functional. Then we replace it with an `EnsembleReducedFunctional`, using the same annotated functional, the controls `[v_p, v_s]`, and the ensemble communicator `wave_guess.comm`.

To monitor optimization runs under MPI, the notebook uses callbacks provided by `EnsembleReducedFunctional`: `record_local_functional` prints the local value on the root member, and `save_iteration` writes the current models to VTK on ensemble rank 0.

In [ ]:
functional_values = []

def record_local_functional(value, controls):
    functional_values.append(float(value))
    if wave_guess.comm.comm.rank == 0 and wave_guess.comm.ensemble_comm.rank == 0:
        print(f"Local J = {float(value):.6e}", flush=True)
    return value


def save_iteration(functional, gradient, controls):
    if WRITE_VTK_OUTPUT and wave_guess.comm.ensemble_comm.rank == 0:
        vp_current = fire.Function(vp_guess.function_space(), name="vp")
        vs_current = fire.Function(vs_guess.function_space(), name="vs")
        vp_current.assign(controls[0])
        vs_current.assign(controls[1])
        iteration = len(functional_values)
        write_vtk_on_ensemble_root(
            wave_guess, output_dir / f"vp_iter_{iteration:03d}.pvd", vp_current
        )
        write_vtk_on_ensemble_root(
            wave_guess, output_dir / f"vs_iter_{iteration:03d}.pvd", vs_current
        )
    return gradient
local_J_hat = wave_guess.automated_adjoint.create_reduced_functional(
    wave_guess.functional_value
)
controls = wave_guess.automated_adjoint.controls

J_hat = EnsembleReducedFunctional(
    wave_guess.functional_value,
    [Control(controls[0]), Control(controls[1])],
    wave_guess.comm,
    eval_cb_post=record_local_functional,
    derivative_cb_post=save_iteration,
)

wave_guess.automated_adjoint.reduced_functional = J_hat


## 7.5 Smoke test: one functional evaluation and one gradient

Before calling ROL, which may trigger many evaluations, we validate the pipeline with one call to `J` and one call to the gradient. This measures the baseline cost of an iteration and confirms that the ensemble parallelism is reducing across sources. `J_total` should already be the sum of the local source contributions.

In [ ]:
SMOKE_TEST_BEFORE_ROL = True
RUN_OPTIMIZATION = True

if SMOKE_TEST_BEFORE_ROL:
    current_values = [vp_guess, vs_guess]

    _t0 = time.perf_counter()
    J_initial = J_hat(current_values)
    _t_J = time.perf_counter() - _t0

    _t0 = time.perf_counter()
    grad_initial = J_hat.derivative(apply_riesz=True)
    _t_g = time.perf_counter() - _t0

    if world.rank == 0:
        try:
            J_val = float(J_initial)
        except (TypeError, ValueError):
            J_val = J_initial
        if isinstance(grad_initial, (list, tuple)):
            grad_norms = [fire.norm(g) for g in grad_initial]
        else:
            grad_norms = [fire.norm(grad_initial)]
        print(f"\n[smoke] initial J_total = {J_val}")
        print(f"[smoke] ||dJ/dvp||, ||dJ/dvs|| = {grad_norms}")
        print(f"[smoke] J time = {_t_J:.2f}s, gradient time = {_t_g:.2f}s")
        # Reset counters and history so the optimization starts cleanly.
    J_hat.functional_evaluations = 0
    J_hat.gradient_evaluations = 0
    functional_values.clear()



RUN_TAYLOR_TEST = False

if RUN_TAYLOR_TEST:
    directions = [
        fire.Function(vp_guess.function_space()).assign(0.01),
        fire.Function(vs_guess.function_space()).assign(0.005),
    ]
    rate = taylor_test(J_hat, controls, directions)
    if wave_guess.comm.comm.rank == 0 and wave_guess.comm.ensemble_comm.rank == 0:
        print(f"Taylor test rate: {rate:.4f}")
    assert rate > 1.9


## 9. Minimization problem with ROL

We pass the `EnsembleReducedFunctional` to `MinimizationProblem` and use `ROLSolver`. Bounds are defined separately for `v_p` and `v_s`.

In [ ]:
vp_bounds = (2.0, 3.5)
vs_bounds = (1.1, 1.90)
maxiter = 30

problem = MinimizationProblem(J_hat, bounds=[vp_bounds, vs_bounds])

rol_parameters = {
    "General": {
        "Secant": {
            "Type": "Limited-Memory BFGS",
            "Maximum Storage": 10,
        },
    },
    "Step": {
        "Type": "Line Search",
        "Line Search": {
            "Descent Method": {
                "Type": "Quasi-Newton Step",
            },
        },
    },
    "Status Test": {
        "Gradient Tolerance": 1.0e-8,
        "Step Tolerance": 1.0e-12,
        "Iteration Limit": maxiter,
    },
}

if RUN_OPTIMIZATION:
    solver = ROLSolver(problem, rol_parameters, inner_product="L2")
    _t0 = time.perf_counter()
    vp_optimised, vs_optimised = solver.solve()
    _t_rol = time.perf_counter() - _t0
    if world.rank == 0:
        print(f"\n[timing] ROL total = {_t_rol:.1f}s, "
              f"J evals = {J_hat.functional_evaluations}, "
              f"grad evals = {J_hat.gradient_evaluations}")

    if WRITE_VTK_OUTPUT and wave_guess.comm.ensemble_comm.rank == 0:
        vp_optimised.rename("vp_optimised")
        vs_optimised.rename("vs_optimised")
        write_vtk_on_ensemble_root(
            wave_guess, output_dir / "vp_optimised.pvd", vp_optimised
        )
        write_vtk_on_ensemble_root(
            wave_guess, output_dir / "vs_optimised.pvd", vs_optimised
        )

    if wave_guess.comm.comm.rank == 0 and wave_guess.comm.ensemble_comm.rank == 0:
        print("Optimized models saved in", output_dir)
else:
    if world.rank == 0:
        print("[FWI] RUN_OPTIMIZATION = False, skipping ROLSolver.solve().")


## 10. Local functional history

The `record_local_functional` callback collects the local values returned by the reduced functional on each ensemble member. `EnsembleReducedFunctional` sums these values internally for ROL.

In [ ]:
if wave_guess.comm.comm.rank == 0 and wave_guess.comm.ensemble_comm.rank == 0:
    print("Local functional history on the root member:")
    for i, value in enumerate(functional_values):
        print(f"  {i}: {value:.6e}")

    if functional_values:
        plt.figure(figsize=(6, 4))
        plt.semilogy(functional_values, marker="o")
        plt.xlabel("evaluation")
        plt.ylabel("local J")
        plt.title("Elastic FWI with EnsembleReducedFunctional + ROL")
        plt.grid(True)
        plt.tight_layout()
        plt.show()


## Notes

- This example optimizes `v_p` and `v_s`; `rho` is fixed.
- `IsotropicWave` internally converts `{rho, v_p, v_s}` into Lame parameters when assembling the variational form.
- The observed data are synthetic, generated with the same discretization but with a different horizontal layered elastic model.
- For a more robust inversion, increase `maxiter`, reduce `edge_length`, use a multiscale frequency strategy, and review the update region near the surface.
- The important path in this notebook is the composition: `AutomatedAdjoint` records the local solve, `EnsembleReducedFunctional` sums the sources, and ROL solves the optimization problem in `v_p` and `v_s`.